# Phase 1b: Tokenizers & Embeddings

**Goal:** Understand how text gets converted into numbers that models can work with.

Two steps happen before any model processing:
1. **Tokenization** — split text into pieces (tokens)
2. **Embedding** — convert each token into a vector that captures meaning

This notebook lets you experiment with both hands-on.

In [ ]:
from transformers import AutoTokenizer
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

---
## 1. Why Tokenization? Why Not Just Split on Spaces?

The naive approach is to split text by spaces. Let's see why that fails.

In [ ]:
text = "I can't believe OpenAI's ChatGPT doesn't understand unhappiness!"

# Naive: split on spaces
naive_tokens = text.split()
print("Space-split tokens:")
for i, t in enumerate(naive_tokens):
    print(f"  {i}: '{t}'")

print(f"\nProblems:")
print(f"  - \"can't\" is one token — should it be \"can\" + \"not\"?")
print(f"  - \"OpenAI's\" mixes a name with punctuation")
print(f"  - \"unhappiness\" — the model can't see \"un\" + \"happy\" + \"ness\"")
print(f"  - What about languages without spaces (Chinese, Japanese)?")
print(f"  - What about typos or rare words the model has never seen?")

---
## 2. Three Tokenizer Types

Modern models use **subword tokenization** — they break words into pieces that balance vocabulary size with coverage.

| Tokenizer | Used by | How it works |
|---|---|---|
| **BPE** (Byte-Pair Encoding) | GPT-2, GPT-4, LLaMA | Starts with characters, merges most frequent pairs |
| **WordPiece** | BERT, RoBERTa | Similar to BPE but uses likelihood instead of frequency |
| **SentencePiece** | T5, FLAN-T5 | Language-agnostic, treats text as raw bytes (no pre-tokenization) |

Let's see all three on the **same text**.

In [ ]:
# Load tokenizers from real models
tokenizers = {
    "GPT-2 (BPE)": AutoTokenizer.from_pretrained("gpt2"),
    "BERT (WordPiece)": AutoTokenizer.from_pretrained("bert-base-uncased"),
    "T5 (SentencePiece)": AutoTokenizer.from_pretrained("t5-small"),
}

text = "The unhappy customer couldn't understand tokenization"

print(f"Input: \"{text}\"\n")
print(f"{'Tokenizer':<25} {'# Tokens':<10} {'Tokens'}")
print("-" * 90)

for name, tok in tokenizers.items():
    tokens = tok.tokenize(text)
    print(f"{name:<25} {len(tokens):<10} {tokens}")

In [ ]:
# Let's look more closely at how each one handles subwords

word = "unhappiness"
print(f"How each tokenizer splits \"{word}\":\n")

for name, tok in tokenizers.items():
    tokens = tok.tokenize(word)
    ids = tok.encode(word, add_special_tokens=False)
    print(f"  {name:<25} → {tokens}")
    print(f"  {'':25}   IDs: {ids}\n")

print("Notice:")
print("  - BERT uses '##' prefix to mark continuation subwords")
print("  - GPT-2 uses 'Ġ' prefix to mark word beginnings")
print("  - T5 uses '▁' (underscore) to mark word beginnings")
print("\nDifferent conventions, same idea: break rare words into known pieces.")

In [ ]:
# The full pipeline: text → tokens → IDs → back to text

tok = tokenizers["GPT-2 (BPE)"]
text = "Machine learning is fascinating!"

# Step 1: Tokenize (text → tokens)
tokens = tok.tokenize(text)
print(f"1. Text:    '{text}'")
print(f"2. Tokens:  {tokens}")

# Step 2: Encode (tokens → integer IDs)
ids = tok.encode(text)
print(f"3. IDs:     {ids}")

# Step 3: Decode (IDs → text)
decoded = tok.decode(ids)
print(f"4. Decoded: '{decoded}'")

print(f"\nVocab size: {tok.vocab_size:,} tokens")
print(f"\nThe model never sees text — only these integer IDs.")
print(f"Each ID maps to a row in the embedding matrix (next section).")

### YOUR UNDERSTANDING

*Why do models use subword tokenization instead of whole words or individual characters?*

> (your notes here)

*What would be the problem with a character-level tokenizer?*

> (your notes here)


---
## 3. Vocabulary Size and Its Trade-offs

Vocabulary size is a fundamental design choice. Let's see why it matters.

In [ ]:
# Compare vocabulary sizes
print(f"{'Model':<25} {'Vocab Size':<15} {'Type'}")
print("-" * 55)
for name, tok in tokenizers.items():
    print(f"{name:<25} {tok.vocab_size:<15,} {type(tok).__name__}")

print(f"\nTrade-off:")
print(f"  Larger vocab  → fewer tokens per sentence → faster processing")
print(f"                → but bigger embedding table → more memory")
print(f"  Smaller vocab → more tokens per sentence → slower processing")
print(f"                → but smaller embedding table → less memory")

In [ ]:
# See the trade-off in action: how many tokens for the same text?

long_text = (
    "Customer support automation using large language models can significantly "
    "reduce response times and improve satisfaction scores. The system analyzes "
    "incoming tickets, categorizes them by urgency, and generates appropriate "
    "responses using retrieval-augmented generation techniques."
)

print(f"Text length: {len(long_text)} characters\n")

for name, tok in tokenizers.items():
    tokens = tok.tokenize(long_text)
    print(f"{name:<25} → {len(tokens)} tokens")

print(f"\nFewer tokens = less computation in self-attention (remember: n² cost)")

In [ ]:
# Special tokens — each model adds its own control tokens

text = "Hello world"

for name, tok in tokenizers.items():
    ids = tok.encode(text)
    decoded_tokens = [tok.decode([id]) for id in ids]
    print(f"{name}:")
    print(f"  IDs:    {ids}")
    print(f"  Tokens: {decoded_tokens}")
    if tok.special_tokens_map:
        specials = {k: v for k, v in tok.special_tokens_map.items() if isinstance(v, str)}
        print(f"  Special tokens: {specials}")
    print()

---
## 4. From Token IDs to Embeddings

Now we know how text becomes integer IDs. But models need **vectors**, not integers.

The **embedding layer** is a lookup table:
- Row 0 → vector for token ID 0
- Row 1 → vector for token ID 1
- ...
- Row 50256 → vector for token ID 50256

These vectors are **learned during training** — the model adjusts them so that similar words end up with similar vectors.

In [ ]:
# Build an embedding layer from scratch to see how it works

vocab_size = 10  # tiny vocab for illustration
embedding_dim = 4  # each token gets a 4-dimensional vector

# The embedding table: vocab_size rows × embedding_dim columns
embedding = torch.nn.Embedding(vocab_size, embedding_dim)

print("Embedding table (randomly initialized):")
print(f"Shape: {embedding.weight.shape} (vocab_size × embedding_dim)\n")

for i in range(vocab_size):
    vec = embedding.weight[i].detach().numpy()
    print(f"  Token ID {i} → [{', '.join(f'{v:+.3f}' for v in vec)}]")

print(f"\nLooking up token ID 3:")
result = embedding(torch.tensor([3]))
print(f"  {result.detach().numpy()[0]}")
print(f"  Same as row 3 above — it's just a lookup!")

In [ ]:
# Now let's see real embeddings from GPT-2

from transformers import GPT2Model

model = GPT2Model.from_pretrained("gpt2")
tok = tokenizers["GPT-2 (BPE)"]

# The embedding layer
embed_layer = model.wte  # wte = word token embeddings
print(f"GPT-2 embedding table:")
print(f"  Shape: {embed_layer.weight.shape}")
print(f"  Vocab size: {embed_layer.weight.shape[0]:,}")
print(f"  Embedding dim: {embed_layer.weight.shape[1]}")
print(f"  Total parameters: {embed_layer.weight.numel():,}")
print(f"\n  That's {embed_layer.weight.numel() * 4 / 1024 / 1024:.1f} MB just for the embedding table!")

In [ ]:
# The full journey: text → tokens → IDs → embeddings

text = "The cat sat"
print(f"Input: '{text}'\n")

# Tokenize
tokens = tok.tokenize(text)
print(f"Step 1 - Tokenize:  {tokens}")

# To IDs
ids = tok.encode(text)
print(f"Step 2 - To IDs:    {ids}")

# To embeddings
id_tensor = torch.tensor([ids])
with torch.no_grad():
    embeds = embed_layer(id_tensor)

print(f"Step 3 - Embeddings shape: {embeds.shape}")
print(f"         (1 batch × {len(ids)} tokens × 768 dimensions)")
print(f"\nFirst 8 values of each token's embedding:")
for token, emb in zip(tokens, embeds[0]):
    vals = ', '.join(f'{v:.3f}' for v in emb[:8].numpy())
    print(f"  '{token}': [{vals}, ...]")

print(f"\nThese vectors are what the model actually processes.")
print(f"Everything from Phase 1 (attention, etc.) operates on these.")

### YOUR UNDERSTANDING

*What is the relationship between token IDs and embeddings?*

> (your notes here)

*Why are embeddings vectors and not just single numbers?*

> (your notes here)


---
## 5. Embeddings Capture Meaning

The magic of learned embeddings: **similar words end up close together in vector space**.

Let's see this with real model embeddings.

In [ ]:
# Get embeddings for a set of words and measure similarity

words = [
    # Animals
    "cat", "dog", "fish", "bird",
    # Colors
    "red", "blue", "green", "yellow",
    # Countries
    "France", "Germany", "Japan", "Brazil",
    # Tech
    "Python", "Java", "code", "software",
]

def get_embedding(word, tokenizer, embed_layer):
    """Get the static embedding for a word (average if multi-token)."""
    ids = tokenizer.encode(word, add_special_tokens=False)
    id_tensor = torch.tensor([ids])
    with torch.no_grad():
        emb = embed_layer(id_tensor)
    return emb[0].mean(dim=0).numpy()  # average if word splits into multiple tokens

embeddings = {word: get_embedding(word, tok, embed_layer) for word in words}

# Compute cosine similarity between all pairs
emb_matrix = np.stack([embeddings[w] for w in words])
sim_matrix = cosine_similarity(emb_matrix)

# Plot similarity heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='RdYlBu_r', vmin=-0.2, vmax=1.0)
ax.set_xticks(range(len(words)))
ax.set_xticklabels(words, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words, fontsize=9)
ax.set_title('Cosine Similarity Between Word Embeddings (GPT-2)', fontsize=12)
plt.colorbar(im, ax=ax, label='Cosine Similarity')

# Add category separators
for pos in [4, 8, 12]:
    ax.axhline(y=pos - 0.5, color='black', linewidth=1.5)
    ax.axvline(x=pos - 0.5, color='black', linewidth=1.5)

plt.tight_layout()
plt.show()

print("Look for brighter blocks along the diagonal — words in the same")
print("category should be more similar to each other than to other categories.")
print("\nNote: These are STATIC embeddings (pre-attention). After self-attention,")
print("the contextual embeddings would show even clearer semantic groupings.")

In [ ]:
# Visualize embeddings in 2D using PCA

pca = PCA(n_components=2)
coords = pca.fit_transform(emb_matrix)

# Color by category
categories = {
    "Animals": ["cat", "dog", "fish", "bird"],
    "Colors": ["red", "blue", "green", "yellow"],
    "Countries": ["France", "Germany", "Japan", "Brazil"],
    "Tech": ["Python", "Java", "code", "software"],
}

colors = {"Animals": "#e74c3c", "Colors": "#3498db", "Countries": "#2ecc71", "Tech": "#9b59b6"}

fig, ax = plt.subplots(figsize=(10, 7))

for cat_name, cat_words in categories.items():
    for word in cat_words:
        idx = words.index(word)
        ax.scatter(coords[idx, 0], coords[idx, 1], c=colors[cat_name],
                   s=100, zorder=5, edgecolors='black', linewidth=0.5)
        ax.annotate(word, (coords[idx, 0], coords[idx, 1]),
                    textcoords="offset points", xytext=(8, 5), fontsize=10)

# Legend
for cat_name, color in colors.items():
    ax.scatter([], [], c=color, s=100, label=cat_name, edgecolors='black', linewidth=0.5)
ax.legend(fontsize=10, loc='best')

ax.set_title('Word Embeddings in 2D (PCA projection)', fontsize=13)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Words from the same category should cluster together.")
print("The clustering may not be perfect — we're squeezing 768 dimensions into 2.")
print("But the tendency should be visible.")

In [ ]:
# The classic word embedding test: analogies
# king - man + woman ≈ queen (in theory)
# Let's try with GPT-2's embeddings

def analogy(a, b, c, embeddings, top_n=5):
    """
    a is to b as c is to ???
    Computes: b - a + c, finds nearest neighbor
    """
    target = embeddings[b] - embeddings[a] + embeddings[c]
    
    results = []
    for word, emb in embeddings.items():
        if word in (a, b, c):
            continue
        sim = cosine_similarity([target], [emb])[0][0]
        results.append((word, sim))
    
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_n]

# Build a larger embedding set for analogies
analogy_words = [
    "king", "queen", "man", "woman", "boy", "girl",
    "France", "Paris", "Germany", "Berlin", "Japan", "Tokyo",
    "cat", "kitten", "dog", "puppy",
    "big", "bigger", "small", "smaller",
    "good", "better", "bad", "worse",
]

analogy_embeddings = {w: get_embedding(w, tok, embed_layer) for w in analogy_words}

tests = [
    ("man", "king", "woman", "queen"),
    ("France", "Paris", "Germany", "Berlin"),
    ("cat", "kitten", "dog", "puppy"),
    ("good", "better", "bad", "worse"),
]

print("Word Analogy Tests (a → b, c → ???)\n")
for a, b, c, expected in tests:
    results = analogy(a, b, c, analogy_embeddings, top_n=3)
    top_word = results[0][0]
    match = "✓" if top_word == expected else "✗"
    print(f"  {a} → {b}, {c} → ???  (expected: {expected})")
    for word, sim in results:
        indicator = " ←" if word == expected else ""
        print(f"    {word}: {sim:.4f}{indicator}")
    print()

print("Note: GPT-2's static embeddings may not ace all analogies.")
print("Word2Vec/GloVe were specifically trained for this.")
print("GPT-2 embeddings are just the starting point — before any attention layers.")

### YOUR UNDERSTANDING

*Why do similar words end up with similar embeddings?*

> (your notes here)

*What's the difference between static embeddings (what we see here) and contextual embeddings (after attention)?*

> (your notes here)


---
## 6. Contextual Embeddings: Before vs After Attention

The embeddings above are **static** — the same word always gets the same vector.

After passing through the Transformer layers, each word's embedding becomes **contextual** — shaped by the surrounding words. Let's see the difference.

In [ ]:
# The word "bank" in two different contexts

sentences = [
    "I deposited money at the bank today",
    "We had a picnic on the river bank",
]

print("Comparing 'bank' in two contexts:\n")

static_embeddings = []
contextual_embeddings = []

for sent in sentences:
    inputs = tok(sent, return_tensors="pt")
    tokens = tok.tokenize(sent)
    
    # Find where "bank" is
    bank_idx = tokens.index("bank") if "bank" in tokens else tokens.index("Ġbank")
    
    with torch.no_grad():
        # Static: just the embedding lookup
        static = embed_layer(inputs["input_ids"])[0][bank_idx]
        
        # Contextual: after all 12 transformer layers
        outputs = model(**inputs)
        contextual = outputs.last_hidden_state[0][bank_idx]
    
    static_embeddings.append(static.numpy())
    contextual_embeddings.append(contextual.numpy())
    
    print(f"  \"{sent}\"")
    print(f"  'bank' is token #{bank_idx}: '{tokens[bank_idx]}'\n")

# Compare
static_sim = cosine_similarity([static_embeddings[0]], [static_embeddings[1]])[0][0]
contextual_sim = cosine_similarity([contextual_embeddings[0]], [contextual_embeddings[1]])[0][0]

print(f"Cosine similarity of 'bank' between the two sentences:")
print(f"  Static embeddings (before attention):  {static_sim:.4f}")
print(f"  Contextual embeddings (after attention): {contextual_sim:.4f}")
print(f"\n  Static: identical (same word = same vector, regardless of context)")
print(f"  Contextual: different (attention mixed in surrounding context)")
print(f"\n  The drop in similarity shows the model distinguishes")
print(f"  financial 'bank' from river 'bank' after processing context.")

### YOUR UNDERSTANDING

*Why are contextual embeddings more useful than static embeddings for NLP tasks?*

> (your notes here)


---
## 7. Token Count Matters: Why You Care About Tokenization

In practice, tokenization directly affects **cost and capability**:
- LLM APIs charge **per token** (not per word or character)
- Context windows are measured in **tokens**
- More tokens = more self-attention computation (n² cost)

In [ ]:
# Real-world cost implications

gpt2_tok = tokenizers["GPT-2 (BPE)"]

examples = [
    "Hi",
    "Hello, how are you doing today?",
    "The customer reported that their order #A12345 was delayed by 3 business days and requested an immediate refund of $249.99 to their credit card ending in 4532.",
    "pneumonoultramicroscopicsilicovolcanoconiosis",  # longest English word
    "こんにちは世界",  # "Hello world" in Japanese
    "🎉🚀💻",  # emojis
]

print(f"{'Text (truncated)':<55} {'Chars':<8} {'Tokens':<8} {'Ratio'}")
print("-" * 85)

for text in examples:
    tokens = gpt2_tok.tokenize(text)
    display = text[:50] + "..." if len(text) > 50 else text
    ratio = len(tokens) / len(text.split()) if text.split() else 0
    print(f"{display:<55} {len(text):<8} {len(tokens):<8} {ratio:.1f} tok/word")

print(f"\nNotice:")
print(f"  - Common English ≈ 1.3 tokens per word")
print(f"  - Rare/long words get split into many tokens")
print(f"  - Non-English text uses MORE tokens (less efficient)")
print(f"  - Emojis and special characters can be expensive")
print(f"\nAt GPT-4 pricing (~$0.01/1K tokens), this matters at scale.")

---
## 8. Summary

```
Raw text: "The unhappy customer"
     ↓
Tokenizer splits into subwords: ["The", "un", "##happy", "customer"]
     ↓
Maps to integer IDs: [1996, 4895, 18218, 8619]
     ↓
Embedding lookup → one vector per token (e.g., 768 dims each)
     ↓
Add positional encoding → model knows token order
     ↓
Feed into Transformer layers → static embeddings become contextual
```

### Key Takeaways
1. **Tokenization** is not word-splitting — it's subword decomposition
2. **Different models use different tokenizers** — same text, different tokens
3. **Embeddings** are learned vectors that capture meaning
4. **Static embeddings** are the starting point; **contextual embeddings** (after attention) are where the real power is
5. **Token count** directly impacts cost, speed, and context window usage

### YOUR UNDERSTANDING — Final Reflection

*Trace the full journey of a sentence from raw text to model output — what happens at each step?*

> (your notes here)

*If you were building a customer support system, why would tokenization choices matter?*

> (your notes here)


---
## Next Up: Phase 2 — Text Generation with LLMs

Now that you understand how text becomes vectors, the next notebook will show how models **generate** text from those vectors — using GPT-2 to experiment with different generation strategies (greedy, beam search, sampling).

**Before moving on**, make sure you've filled in all the `YOUR UNDERSTANDING` sections above!